In [1]:
import os
import sys
import logging

import polars as pl

from pathlib import Path
from dotenv import load_dotenv
from ollama import chat
from opendataloader_pdf import convert

# Env
pl.Config.set_tbl_rows(10)
load_dotenv("../.env")
INSTALL_PATH = os.environ["INSTALL_PATH"]
URL_AF = os.environ["API_URL_ALPHAFOLD_STRUCTPRED"]
sys.path.append(str(Path(INSTALL_PATH).resolve()))

# Modules
from modules.pymol_align import get_alphafold_structures, pymol_align_pairs

# Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s:%(name)s:%(message)s"
)
logger = logging.getLogger(__name__)

In [2]:
# ────────────────────────────────────────────────────────────
#      Args
# ────────────────────────────────────────────────────────────

PDF_DIR = f"{INSTALL_PATH}/data_sync/goldStandard_geneList"
OUT_DIR_PARSE = f"{INSTALL_PATH}/data_sync/goldStandard_geneList/opendataloader_out"

In [3]:
# ────────────────────────────────────────────────────────────
#      In
# ────────────────────────────────────────────────────────────

out_dir_parse = Path(OUT_DIR_PARSE)
out_dir_parse.mkdir(parents=True, exist_ok=True)

pdf_dir = Path(PDF_DIR)
df_pdf_names = pl.DataFrame({
    "file_path": [str(p) for p in pdf_dir.glob("*.pdf")],
    "file_contents": ["" for _ in pdf_dir.glob("*.pdf")]
})

df_pdf_names

file_path,file_contents
str,str
"""/home/juliandeanmoran/___git_r…",""""""
"""/home/juliandeanmoran/___git_r…",""""""
"""/home/juliandeanmoran/___git_r…",""""""
"""/home/juliandeanmoran/___git_r…",""""""
"""/home/juliandeanmoran/___git_r…",""""""
…,…
"""/home/juliandeanmoran/___git_r…",""""""
"""/home/juliandeanmoran/___git_r…",""""""
"""/home/juliandeanmoran/___git_r…",""""""


In [4]:
# ────────────────────────────────────────────────────────────
#      OpenDataLoader: Parse
# ────────────────────────────────────────────────────────────

i = 0

_ = convert(
    input_path=df_pdf_names["file_path"][0],
    output_dir=out_dir_parse,
    format="json"
)

_

Mar 18, 2026 3:09:37 PM org.opendataloader.pdf.processors.DocumentProcessor preprocessing
INFO: File name: /home/juliandeanmoran/___git_repos/BISSH/data_sync/goldStandard_geneList/2022_Tesson.pdf
Mar 18, 2026 3:09:38 PM org.verapdf.gf.model.factory.chunks.ChunkParser parseString
SEVERE: Missing width of glyph with code 32 in fontCEICPG+AdvOTea1a7398
Mar 18, 2026 3:09:38 PM org.verapdf.gf.model.factory.chunks.ChunkParser parseString
SEVERE: Missing width of glyph with code 32 in fontCEICPG+AdvOTea1a7398
Mar 18, 2026 3:09:38 PM org.verapdf.gf.model.factory.chunks.ChunkParser parseString
SEVERE: Missing width of glyph with code 32 in fontCEICPG+AdvOTea1a7398
Mar 18, 2026 3:09:38 PM org.verapdf.gf.model.factory.chunks.ChunkParser parseString
SEVERE: Missing width of glyph with code 32 in fontCEICPG+AdvOTea1a7398
Mar 18, 2026 3:09:38 PM org.verapdf.gf.model.factory.chunks.ChunkParser parseString
SEVERE: Missing width of glyph with code 32 in fontCEICPG+AdvOTea1a7398
Mar 18, 2026 3:09:38 PM 

In [5]:
# ────────────────────────────────────────────────────────────
#      OLlama: Extract
# ────────────────────────────────────────────────────────────


# Test code
path = Path("/home/juliandeanmoran/___git_repos/BISSH/data_sync/goldStandard_geneList/opendataloader_out/2022_Tesson.json")
text = path.read_text()

prompt = f"""
Extract gene records matching this schema:
Gene_name, Refseq_NCBI_accession, Uniprot_or_uniparc_accession, Bacterial_species, Bacterial_system, Data_source, Data_source_notes
from the following text.

Gene_name is our key; it should always be populated for a given row, and it should always be a bacterial gene.
If you cannot populate a field because the paper does not have it (e.g. paper is missing `Refseq_NCBI_accession`), assign it `NA`.

```text
{text}
```

Return a JSON array of objects with exactly those fields.
"""
response = chat(
    model="llama3.2:1b-instruct-q4_0",
    messages=[{"role": "user", "content": prompt}],
    format="json",
    options={
        "num_ctx": 2048,
        "num_predict": 512,
        "temperature": 0,
    },
)

print(response.message.content)

INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


{
  "type": "array",
  "items": {
    "id": "string",
    "title": "string",
    "author": "string",
    "date": "string",
    "volume": "number",
    "issue": "number",
    "pages": "number",
    "doi": "string"
  }
}


In [7]:
# ────────────────────────────────────────────────────────────
#      OLlama: Extract (chunked)
# ────────────────────────────────────────────────────────────

# def chunk_text(text, max_chars=6000):
#     return [text[i:i+max_chars] for i in range(0, len(text), max_chars)]

# chunks = chunk_text(text)



chunk = text[:6000]

prompt = f"""
Extract gene records matching this schema:
Gene_name, Refseq_NCBI_accession, Uniprot_or_uniparc_accession, Bacterial_species, Bacterial_system, Data_source, Data_source_notes
from the following text.

Gene_name is our key; it should always be populated for a given row, and it should always be a bacterial gene.
If you cannot populate a field because the paper does not have it, assign it NA.

```text
{chunk}
```

Return a JSON array of objects with exactly those fields.
"""
response = chat(
    model="llama3.2:1b-instruct-q4_0",
    messages=[{"role": "user", "content": prompt}],
    format="json",
    options={
        "num_ctx": 2048,
        "num_predict": 512,
        "temperature": 0,
    },
)

response

INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"


ChatResponse(model='llama3.2:1b-instruct-q4_0', created_at='2026-03-18T19:24:50.171363433Z', done=True, done_reason='length', total_duration=140345139251, load_duration=3648028799, prompt_eval_count=2048, prompt_eval_duration=74759941379, eval_count=512, eval_duration=39297291417, message=Message(role='assistant', content='{\n  "0": {\n    "id": "0",\n    "name": "NATURE COMMUNICATIONS| (2022) 13:2561 |https://doi.org/10.1038/s41467-022-30269-9 |www.nature.com/naturecommunications 1"\n  },\n  "1": {\n    "id": "1",\n    "name": "ARTICLE NATURE COMMUNICATIONS | https://doi.org/10.1038/s41467-022-30269-9",\n    "title": "CRISPR-Cas systems",\n    "author": "Jean Cury, Aude Bernard"\n  },\n  "2": {\n    "id": "2",\n    "name": "ARTICLE NATURE COMMUNICATIONS | https://doi.org/10.1038/s41467-022-30269-9",\n    "title": "Restriction-Modification (RM) and Abortive Infection (Abi)",\n    "author": "Jean Cury, Aude Bernard"\n  },\n  "3": {\n    "id": "3",\n    "name": "ARTICLE NATURE COMMUNICAT

In [6]:
text

'{\n  "file name" : "2022_Tesson.pdf",\n  "number of pages" : 10,\n  "author" : "Florian Tesson",\n  "title" : "Systematic and quantitative view of the antiviral arsenal of prokaryotes",\n  "creation date" : "D:20220504212302+05\'30\'",\n  "modification date" : "D:20241208152800-05\'00\'",\n  "kids" : [ {\n    "type" : "paragraph",\n    "id" : 285,\n    "page number" : 1,\n    "bounding box" : [ 43.597, 640.32, 94.98, 655.704 ],\n    "font" : "AdvOTea1a7398",\n    "font size" : 13.948,\n    "text color" : "[0.0, 0.0, 0.0]",\n    "content" : "ARTICLE"\n  }, {\n    "type" : "paragraph",\n    "id" : 294,\n    "page number" : 1,\n    "bounding box" : [ 46.602, 620.653, 264.122, 635.068 ],\n    "font" : "AdvOTcb88df00",\n    "font size" : 7.97,\n    "text color" : "[1.0, 1.0, 1.0]",\n    "content" : "https://doi.org/10.1038/s41467-022-30269-9 OPEN"\n  }, {\n    "type" : "heading",\n    "id" : 287,\n    "level" : "Subtitle",\n    "page number" : 1,\n    "bounding box" : [ 43.597, 564.657, 52